# 1. COmprobamos el entorno

Verificamos si cuda se esta instalado correctamente en el entorno de ejecución.

In [ ]:
!nvidia-smi
!nvcc --version

Mon May  4 18:25:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 2. Instalación de NVIDIA RAPIDS

Instalamos las librerias de Nvidia Rapids que necesitamos para el procesado y modelado mediante la GPU

In [ ]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


# 3. Importación de librerías

Realizamos la instalación de las librerías que usaremos.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import cudf
from cuml.model_selection import train_test_split
from cuml.linear_model import LogisticRegression
from cuml.metrics import accuracy_score, confusion_matrix

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

# 4. Carga de datos

Cargamos el dataset de sklearn.

In [ ]:
# Cargar dataset
data = fetch_20newsgroups(subset='train')

texts = data.data
labels = data.target

print("Número de documentos:", len(texts))
print("Número de clases:", len(data.target_names))
print("Ejemplo de texto:\n", texts[0][:200], "...")

Número de documentos: 11314
Número de clases: 20
Ejemplo de texto:
 From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out ...


# 5. Procesamiento de datos

Se transforman los textos en variables numéricas usando vectorización y se convierten a estructuras compatibles con NVIDIA RAPIDS.

In [ ]:
# Convertir texto a números
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(texts)

X_cudf = cudf.DataFrame(X.toarray())
y_cudf = cudf.Series(labels)

print("Forma de X_cudf:", X_cudf.shape)
print("Tipo de y_cudf:", type(y_cudf))

Forma de X_cudf: (11314, 5000)
Tipo de y_cudf: <class 'cudf.core.series.Series'>


# 6. División de datos

Dividimos el dataset en 80% entrenamiento y 20% tests.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cudf, y_cudf, test_size=0.2, random_state=42
)

print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)

X_train.shape: (9051, 5000)
X_test.shape: (2263, 5000)


# 7. Modelo con RAPIDS

Entrenamos el modelo de clasificación con regresión logistica con la libreria de nvidia rapids, operando integramente en GPU y evaluado la precision y el reporte de clasificación.

In [ ]:
# Crear y entrenar modelo
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy en test:", acc)

Accuracy en test: 0.8519664162615996


In [ ]:
# necesitamos CPU para classification_report
y_test_cpu = y_test.to_pandas().values  # o .to_numpy() según tu versión
y_pred_cpu = y_pred.to_pandas().values

print("\nClassification Report:")
print(classification_report(y_test_cpu, y_pred_cpu))


Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.91      0.92        97
           1       0.63      0.75      0.68       104
           2       0.80      0.77      0.78       115
           3       0.71      0.75      0.73       123
           4       0.82      0.71      0.76       126
           5       0.81      0.81      0.81       106
           6       0.82      0.83      0.82       109
           7       0.83      0.86      0.85       139
           8       0.89      0.89      0.89       122
           9       0.88      0.96      0.92       102
          10       0.94      0.91      0.92       108
          11       0.99      0.94      0.97       125
          12       0.77      0.79      0.78       114
          13       0.89      0.92      0.91       119
          14       0.96      0.91      0.93       127
          15       0.87      0.88      0.87       122
          16       0.90      0.92      0.91       121
   